# DemandIQ — Chapter 4: Results

Renders every headline table **from the run manifest and stage artifacts**
(`model/artifacts/runs/latest/`), never from hand-copied numbers — the same
provenance rule the pipeline enforces: any figure in a report traces to a
stage, a seed, and a library set.

Regenerate the artifacts with `make train` (+ `make train-full` for the
rolling-origin backtest), or pull a remote run with `make kaggle-pull`.

In [12]:
import json
from pathlib import Path

import joblib
import numpy as np
import pandas as pd

CANDIDATES = [
    Path("../artifacts/runs/latest"),
    Path("artifacts/runs/latest"),
    Path("artifacts/runs/kaggle/artifacts/runs/kaggle"),
]
ARTIFACTS = next((c for c in CANDIDATES if (c / "manifest.json").exists()), None)
assert ARTIFACTS, "no manifest.json found - run `make train` first"
print("artifacts:", ARTIFACTS.resolve())

MANIFEST = json.loads((ARTIFACTS / "manifest.json").read_text())
STAGES = MANIFEST["stages"] if "stages" in MANIFEST else MANIFEST


# Render provenance-derived prose; works in Jupyter/Colab and plain Python.
try:
    from IPython.display import Markdown, display
except ImportError:  # plain-python execution
    Markdown = str

    def display(x):  # type: ignore[misc]
        print(x)


def reading(text):
    display(Markdown(text))

FileNotFoundError: [Errno 2] No such file or directory

### Reading this chapter

Each section below answers one research question with the figure the proposal
promised, then the chart that makes it visible. The prose mirrors
`CHAPTER4_DRAFT.md` (regenerate with `make chapter4`); both read the same
manifest, so they cannot disagree.

In [ ]:
provenance = pd.DataFrame(
    [
        {
            "stage": name,
            "duration_s": round(rec.get("duration_s", float("nan")), 1),
            "python": rec.get("python_version"),
            "seed": rec.get("seed"),
        }
        for name, rec in sorted(STAGES.items())
    ]
)
# Model stages must share one seed; descriptive stages (e.g. the URBAN-WASTE
# summary, seed 0) do not touch model training and are reported separately.
model_stages = provenance["stage"] != "waste_summary"
seeds = provenance.loc[model_stages, "seed"].dropna().unique()
assert len(seeds) <= 1, f"mixed seeds across model stages: {seeds}"
print(f"training seed: {seeds[0] if len(seeds) else 'n/a'} (single-seed model run)")
other = provenance.loc[~model_stages, "stage"].tolist()
if other:
    print("non-seeded descriptive stages (do not affect model metrics):", other)
provenance

## The training data - engineering and visualization (S3.4-3.11)

Every number plotted below is read from the same run that produced RQ1-RQ3:
`features.parquet` (the leakage-safe frame) and `features_meta.joblib` (its
version and column contract). Nothing is re-sampled or hand-entered, so these
figures cannot drift from the trained model.

**Data engineering, end to end.** Four Recruit CSVs are loaded with pinned
dtypes (`ingest`), passed through the S3.7.1 audit (`audit`), and turned into a
26-feature frame (`features`). The rule enforced structurally - by `shift(1)`
rolling windows and a poison-probe rebuild - is that a feature for day `t` may
only read demand strictly before `t`. The target `visitors` is never a feature.


In [ ]:
import matplotlib.pyplot as plt

features = pd.read_parquet(ARTIFACTS / "features.parquet")
meta = joblib.load(ARTIFACTS / "features_meta.joblib")
FEATS, TARGET = meta["feature_columns"], meta["target_column"]
SPLIT = pd.Timestamp("2017-02-01")

n_train = int((features["visit_date"] < SPLIT).sum())
n_test = int((features["visit_date"] >= SPLIT).sum())
print(f"feature set {meta['feature_set_version']}: {features.shape[0]:,} rows x {features.shape[1]} columns")
print(f"{features['air_store_id'].nunique():,} stores | {features['visit_date'].min().date()} -> {features['visit_date'].max().date()}")
print(f"chronological split at {SPLIT.date()}: train {n_train:,} | test {n_test:,} rows (no shuffle, no leakage)")

fig, axes = plt.subplots(1, 3, figsize=(13, 3.6))
axes[0].hist(features[TARGET], bins=60)
axes[0].set_title("raw visitors: right-skewed")
axes[0].set_xlabel("visitors/day")
axes[1].hist(np.log1p(features[TARGET]), bins=60)
axes[1].set_title("log1p(visitors): the training target")
axes[1].set_xlabel("log1p(visitors)")
by_dow = features.groupby("dow")[TARGET].mean()
axes[2].bar(by_dow.index, by_dow.values)
axes[2].set_title("mean demand by weekday")
axes[2].set_xlabel("day of week (0 = Monday)")
fig.suptitle("Figure A.1: the target distribution and the weekly rhythm it must model", y=1.03)
fig.tight_layout()
plt.show()

In [ ]:
import matplotlib.pyplot as plt

corr = features[FEATS].corr(method="spearman")
fig, ax = plt.subplots(figsize=(9, 8))
im = ax.imshow(corr, cmap="RdBu_r", vmin=-1, vmax=1)
ax.set_xticks(range(len(FEATS)), FEATS, rotation=90, fontsize=7)
ax.set_yticks(range(len(FEATS)), FEATS, fontsize=7)
ax.set_title("Figure A.2: Spearman correlation among the 26 leakage-safe features")
fig.colorbar(im, fraction=0.046, pad=0.04)
fig.tight_layout()
plt.show()

stacked = corr.where(~np.eye(len(FEATS), dtype=bool)).stack()
a, b = stacked.index.get_level_values(0), stacked.index.get_level_values(1)
redundant = stacked[a < b].sort_values(ascending=False).head(8)
redundant.index.names = ["feature_a", "feature_b"]
print("most correlated feature pairs (near-duplicates the row/feature subsampling must decorrelate):")
print(redundant.rename("spearman").to_frame().to_string())

associated = (
    features[FEATS + [TARGET]].corr(method="spearman")[TARGET]
    .drop(TARGET).sort_values(key=abs, ascending=False)
)
print("\nfeatures most associated with the target (Spearman):")
associated.head(10).to_frame("spearman")

In [ ]:
import matplotlib.pyplot as plt

null_rows = features[FEATS].isna().sum().sort_values(ascending=False)
present = null_rows[null_rows > 0]
fig, ax = plt.subplots(figsize=(8, 4))
ax.barh(present.index[::-1], present.values[::-1])
ax.set_xlabel("rows with a missing value")
ax.set_title("Figure A.3: missingness is structural, and deliberately never imputed")
fig.tight_layout()
plt.show()

per_store = features.groupby("air_store_id")["visit_date"].count()
print(f"observations per store: median {per_store.median():.0f}, min {per_store.min()}, max {per_store.max()}")
print(f"features carrying nulls: {(null_rows > 0).sum()} of {len(FEATS)} "
      f"(new stores lack history; only ~1/3 of stores have a reservation feed)")
present.to_frame("null_rows")

In [ ]:
reading(f"""**Reading.** `visitors` is strongly right-skewed - median {features[TARGET].median():.0f},
95th percentile {features[TARGET].quantile(0.95):.0f}, maximum {features[TARGET].max():.0f} - exactly
why the model trains on `log1p(visitors)` and is scored with RMSLE as its primary comparison. The
weekday bars show the second structure it exploits: mean demand is highest on weekday index
{int(features.groupby("dow")[TARGET].mean().idxmax())} (0 = Monday), well above the midweek plateau.
Missingness is structural too - {(features[FEATS].isna().sum() > 0).sum()} of {len(FEATS)} features
carry nulls, tracking store age and whether a store has a booking feed, and nothing is ever filled:
boosted trees route a NaN down its own branch by design.""")

## Model architecture - the boosted-tree ensemble (S3.13)

This is a **regression** model over tabular data, so there are no neural layers
and no activation functions. The architecture is a gradient-boosted
decision-tree ensemble: many small trees added one at a time, each fitted to the
residual error the trees before it left behind. The cell below reads the
architecture directly from the saved `lightgbm_model.joblib` /
`catboost_model.joblib` objects rather than from prose.


In [ ]:
import matplotlib.pyplot as plt

lgb = joblib.load(ARTIFACTS / "lightgbm_model.joblib")
print("LightGBM architecture (primary)")
for k in ("objective", "metric", "learning_rate", "num_leaves", "min_data_in_leaf",
          "feature_fraction", "bagging_fraction", "lambda_l2", "deterministic"):
    print(f"  {k:>18}: {lgb.params.get(k)}")
print(f"  {'trees kept':>18}: {lgb.best_iteration_} of {lgb.num_boost_round} ceiling (early stopped, patience 100)")
print(f"  {'target transform':>18}: log1p(visitors) -> expm1, clipped at 0")
print(f"  {'feature set':>18}: {lgb.feature_set_version_}")

imp = pd.Series(lgb.booster_.feature_importance(importance_type="gain"),
                index=lgb.feature_columns_).sort_values()
fig, axes = plt.subplots(1, 2, figsize=(13, 5), gridspec_kw={"width_ratios": [1, 1.3]})
axes[0].barh(imp.index, imp.values)
axes[0].set_title("LightGBM gain importance")
axes[0].set_xlabel("total gain")
axes[1].axis("off")
axes[1].text(0, 0.5,
  "Data -> leakage-safe 26-feature frame\n"
  "      -> LightGBM: 200 trees, lr 0.05, up to 63 leaves\n"
  "         leaf-wise growth, min 40 rows/leaf, L2=1.0,\n"
  "         90% feature + 90% row subsampling\n"
  "      -> log1p space -> expm1, clip at 0 -> point forecast\n"
  "      -> Mondrian adaptive conformal -> 90% interval\n\n"
  "CatBoost: symmetric depth-6 trees, same rows/features/\n"
  "split/target - a second architecture, not a re-run.",
  family="monospace", va="center", fontsize=10)
fig.suptitle("Figure B.1: gradient-boosted regression, not a neural network", y=1.02)
fig.tight_layout()
plt.show()

**Reading.** The "layers" here are **boosting rounds** - 200 trees kept out of a
2000-tree ceiling, the point at which the validation tail stopped improving for
100 consecutive rounds. The activation-function analogue is the **learning rate**
(`0.05`): the fraction of each tree's correction actually added to the ensemble.
Optimisation is least-squares on `log1p` targets (LightGBM `objective=regression`,
CatBoost `loss_function=RMSE`), regularised by `min_data_in_leaf=40`,
`lambda_l2=1.0`, and 90% feature/row subsampling. CatBoost is a genuine second
architecture: it grows **symmetric depth-6** trees where LightGBM grows
**leaf-wise** trees with up to 63 leaves, so the two fail differently - which is
why both are kept.


## Initial performance metrics (S3.12-3.15)

The headline held-out numbers for every model in the run - the three fallback
estimators, the two primaries, and the five baselines - on the same chronological
test window (`2017-02-01` onward), scored exactly once.


In [ ]:
def _metrics(name, m):
    return {"model": name, **{k: m.get(k) for k in ("mae", "rmse", "rmsle", "mase", "bias", "r2")}}

rows = [_metrics(n, m) for n, m in sorted(STAGES["fallbacks"]["metrics"]["fallbacks"].items())]
rows += [_metrics("LightGBM", STAGES["lightgbm"]["metrics"]),
         _metrics("CatBoost", STAGES["catboost"]["metrics"])]
rows += [_metrics(n, m) for n, m in STAGES["baseline"]["metrics"]["baselines"].items()]

perf = pd.DataFrame(rows).sort_values("mae").reset_index(drop=True)
lift = STAGES["lightgbm"]["metrics"]["mae_lift_vs_baseline"]
print(f"LightGBM MAE {STAGES['lightgbm']['metrics']['mae']:.3f} | "
      f"best baseline ({STAGES['baseline']['metrics']['best_baseline']}) "
      f"{STAGES['baseline']['metrics']['best_baseline_mae']:.3f} | "
      f"lift {lift:.1%} | MASE {STAGES['lightgbm']['metrics']['mase']:.3f} (< 1 beats naive)")
perf

### Reading the metrics: continuous error first, classification metrics derived

The primary numbers for this model are the **continuous** ones - MAE, RMSE, RMSLE,
MASE, Bias and R^2 - because `visitors` is a non-negative count: predicting 21 for an
actual 22 is a small error, not a wrong answer, and these are the quantities the
methodology (S3.14-3.15) scores and the model is selected on.

**Accuracy, precision, recall and F1 are classification metrics**: they are defined over
a confusion matrix, which needs a *discrete* label. They are therefore not read off a
continuous forecast directly - but they are not inapplicable either. They become well
defined the moment a **binary event is declared in advance**, and the event this project
turns on is whether a day will be a **high-demand day**. A missed high-demand day is the
unmet-demand (stock-out) risk RQ2 is built around; a false alarm is the surplus risk.

| Question | Continuous metric (primary) | Derived-event metric |
| --- | --- | --- |
| Typical size of the miss | **MAE / RMSE** | - |
| Relative skill on the log scale | **RMSLE** | - |
| Skill vs the naive rule | **MASE** | - |
| Over/under direction | **Bias** | - |
| Variance explained | **R^2** | - |
| Will this be a high-demand day? | - | **accuracy / precision / recall / F1** (next section) |
| Is the prediction interval honest? | - | **coverage** (RQ2 below) |

The classification metrics below are reported on a **declared derived event**, not a
claim that the model is a classifier: the threshold is fixed from the *training* period
alone, so it cannot be tuned to flatter the held-out window.


### Derived-event classification: detecting high-demand days

Define a high-demand ("busy") day as one whose visitor count is at or above the 75th
percentile of the **training** period - the busiest quarter of observed days. The label
comes from the actual count, the prediction from the forecast, so the point forecast can
be scored with a confusion matrix:

| | Actually busy | Actually not busy |
| --- | --- | --- |
| **Forecast busy** | true positive (correctly stocked up) | false positive (over-prepared -> surplus) |
| **Forecast quiet** | false negative (under-prepared -> **unmet demand**) | true negative |

Precision answers "when we prepare for a busy day, how often is it one?"; recall
answers "of the busy days, how many did we see coming?". For food preparation the two
errors are not symmetric - a missed busy day is a stock-out, a false alarm is surplus -
so recall is the metric to watch, and F1 is only a single-number summary of the pair.


In [ ]:
import numpy as np
from sklearn.metrics import (
    accuracy_score,
    confusion_matrix,
    f1_score,
    precision_score,
    recall_score,
)

# Declared threshold: the busiest quarter of the TRAINING period only, so the
# derived label never reads the held-out window it is evaluated on.
TRAIN_VISITORS = features.loc[features["visit_date"] < SPLIT, TARGET]
BUSY_LEVEL = float(np.percentile(TRAIN_VISITORS, 75))
KEYS = ["air_store_id", "visit_date"]

# Score every candidate on the identical set of held-out store-days.
frames = {
    "LightGBM": pd.read_parquet(ARTIFACTS / "lightgbm_predictions.parquet"),
    "CatBoost": pd.read_parquet(ARTIFACTS / "catboost_predictions.parquet"),
    f"baseline ({STAGES['baseline']['metrics']['best_baseline']})": pd.read_parquet(
        ARTIFACTS / "baseline_predictions.parquet"
    ),
}
common = None
for frame in frames.values():
    common = frame[KEYS] if common is None else common.merge(frame[KEYS], on=KEYS)
frames = {name: frame.merge(common, on=KEYS) for name, frame in frames.items()}


def classify(frame, level):
    """Score 'will this be a busy day?' at a declared threshold."""
    y_true = (frame["visitors"] >= level).astype(int)
    y_pred = (frame["prediction"] >= level).astype(int)
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred, labels=[0, 1]).ravel()
    return {
        "prevalence": y_true.mean(),
        "accuracy": accuracy_score(y_true, y_pred),
        "precision": precision_score(y_true, y_pred, zero_division=0),
        "recall": recall_score(y_true, y_pred, zero_division=0),
        "f1": f1_score(y_true, y_pred, zero_division=0),
        "tp": int(tp),
        "fp": int(fp),
        "fn": int(fn),
        "tn": int(tn),
    }


print(
    f"declared busy level: {BUSY_LEVEL:.0f} visitors/day "
    f"(75th percentile of the training period) | {len(common):,} held-out store-days"
)
class_table = (
    pd.DataFrame({name: classify(frame, BUSY_LEVEL) for name, frame in frames.items()})
    .T.rename_axis("model")
.reset_index()[["model", "prevalence", "accuracy", "precision", "recall", "f1", "tp", "fp", "fn", "tn"]]
)
class_table[["tp", "fp", "fn", "tn"]] = class_table[["tp", "fp", "fn", "tn"]].astype(int)
class_table

In [ ]:
# The metrics depend on the declared event, so show the sensitivity rather than one
# number: a rarer event is harder to hit.
levels = {
    "median (50%)": float(np.percentile(TRAIN_VISITORS, 50)),
    "busy (75%)": BUSY_LEVEL,
    "very busy (60)": 60.0,
}
sensitivity = (
    pd.DataFrame({label: classify(frames["LightGBM"], level) for label, level in levels.items()})
    .T.rename_axis("event")
    .reset_index()[["event", "prevalence", "accuracy", "precision", "recall", "f1"]]
)

import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 2, figsize=(12.5, 3.8), gridspec_kw={"width_ratios": [1.05, 1]})
x = np.arange(len(sensitivity))
width = 0.26
for i, metric in enumerate(("precision", "recall", "f1")):
    axes[0].bar(x + (i - 1) * width, sensitivity[metric], width, label=metric)
axes[0].set_xticks(x, sensitivity["event"])
axes[0].set_ylim(0, 1)
axes[0].set_ylabel("score")
axes[0].set_title("Figure C.1: classification metrics depend on the declared event")
axes[0].legend(fontsize=8)

cm = classify(frames["LightGBM"], BUSY_LEVEL)
matrix = np.array([[cm["tn"], cm["fp"]], [cm["fn"], cm["tp"]]])
im = axes[1].imshow(matrix, cmap="Blues")
for (row, col), value in np.ndenumerate(matrix):
    axes[1].text(col, row, f"{value:,}", ha="center", va="center",
                 color="white" if value > matrix.max() / 2 else "black")
axes[1].set_xticks([0, 1], ["forecast quiet", "forecast busy"], fontsize=8)
axes[1].set_yticks([0, 1], ["actually quiet", "actually busy"], fontsize=8)
axes[1].set_title(f"Figure C.2: LightGBM confusion matrix at {BUSY_LEVEL:.0f} visitors")
fig.colorbar(im, ax=axes[1], fraction=0.046, pad=0.04)
fig.tight_layout()
plt.show()
sensitivity

In [ ]:
_lgb = class_table.set_index("model").loc["LightGBM"]
_base = class_table.iloc[-1]
_rare_recall = sensitivity.set_index("event").loc["very busy (60)", "recall"]
reading(f"""**Reading.** At a declared {BUSY_LEVEL:.0f}-visitor/day threshold - the busiest quarter of
the training period, with {_lgb['prevalence']:.0%} of held-out days busy - the continuous forecast
becomes a classifier scoring **accuracy {_lgb['accuracy']:.3f}, precision {_lgb['precision']:.3f},
recall {_lgb['recall']:.3f}, F1 {_lgb['f1']:.3f}**. Accuracy on its own flatters: always predicting
"quiet" already scores {1 - _lgb['prevalence']:.3f}, so precision and recall carry the signal, and
LightGBM beats the best baseline on the pair (F1 {_base['f1']:.3f}) in the same order the continuous
metrics gave.

The asymmetry is the operationally important part. Precision {_lgb['precision']:.3f} sits above recall
{_lgb['recall']:.3f}: the model is more reliable at confirming a busy day than at finding one, so its
{_lgb['fn']:,.0f} false negatives are exactly the days an operator would under-prepare - the
unmet-demand exposure RQ2 quantifies. Those misses concentrate on the rare heavy days, where recall
falls to {_rare_recall:.3f} at a 60-visitor threshold: a point-forecast limitation on rare busy days,
consistent with the interval finding in RQ2 below, not an artefact of the threshold.""")

class_table

## RQ1 — Do models beat the baseline gauntlet? (§3.12–3.15)

In [ ]:
rows = []
base = STAGES["baseline"]["metrics"]
for name, m in base["baselines"].items():
    rows.append({"model": name, **{k: m.get(k) for k in ("mae", "rmse", "rmsle", "mase", "bias")}})
rows.append({"model": "LightGBM", **{k: STAGES["lightgbm"]["metrics"][k] for k in ("mae", "rmse", "rmsle", "mase", "bias")}})
rows.append({"model": "CatBoost", **{k: STAGES["catboost"]["metrics"][k] for k in ("mae", "rmse", "rmsle", "mase", "bias")}})

rq1 = pd.DataFrame(rows).sort_values("mae").reset_index(drop=True)
best_baseline = base["best_baseline"]
lift = STAGES["lightgbm"]["metrics"]["mae_lift_vs_baseline"]
print(f"best baseline: {best_baseline} | LightGBM lift vs best baseline: {lift:.1%}")
rq1

In [ ]:
reading(f"""**Reading.** The weekday-mean baseline is the strongest simple rule - consistent with the
weekday cycle dominating visitor behaviour. Both gradient-boosting models clear it: LightGBM improves
MAE by **{STAGES["lightgbm"]["metrics"]["mae_lift_vs_baseline"]:.1%}** over the best baseline with MASE
{STAGES["lightgbm"]["metrics"]["mase"]:.3f}, far below the 1.0 naive-scale boundary. CatBoost agrees
to within {abs(STAGES["catboost"]["metrics"]["mae_delta_vs_lightgbm"]):.3f} visitors, which strengthens
the claim that the gain comes from the features rather than one library's quirks - the reason Section
3.13 carries a secondary primary at all.""")

In [ ]:
# Chart 4.1 - MAE by model; the dashed line marks the bar a model must clear.
import matplotlib.pyplot as plt

order = rq1.sort_values("mae", ascending=False)
fig, ax = plt.subplots(figsize=(8, 4))
bars = ax.barh(order["model"], order["mae"])
best_row = order.loc[order["model"] == best_baseline, "mae"].iloc[0]
ax.axvline(best_row, linestyle="--", linewidth=1, label=f"best baseline ({best_baseline})")
for b, v in zip(bars, order["mae"]):
    ax.text(v + 0.08, b.get_y() + b.get_height() / 2, f"{v:.2f}", va="center", fontsize=9)
ax.set_xlabel("MAE (visitors) - lower is better")
ax.set_title("Figure 4.1: gradient boosting vs the baseline gauntlet")
ax.legend(loc="lower right")
fig.tight_layout()
plt.show()

## RQ1-sensitivity — Rolling-origin backtest (§3.25)

In [ ]:
folds = pd.read_parquet(ARTIFACTS / "backtest_folds.parquet")
if "backtest" in STAGES:
    bt = STAGES["backtest"]["metrics"]
    print(f"MAE over {bt['n_folds']} folds: {bt['mae_mean']:.3f} ± {bt['mae_std']:.3f} "
          f"| RMSLE {bt['rmsle_mean']:.4f} ± {bt['rmsle_std']:.4f}")
print("single-split LightGBM MAE:", STAGES["lightgbm"]["metrics"]["mae"],
      "- the mean-over-folds matches; the std shows sensitivity to the origin.")
folds

In [ ]:
# Chart 4.2 - per-fold MAE against the mean +/- one standard deviation.
fig, ax = plt.subplots(figsize=(8, 3.5))
ax.plot(folds["fold"], folds["mae"], marker="o", label="fold MAE")
if "backtest" in STAGES:
    ax.axhline(bt["mae_mean"], linestyle="--", linewidth=1, label="mean over folds")
    ax.fill_between(folds["fold"], bt["mae_mean"] - bt["mae_std"], bt["mae_mean"] + bt["mae_std"], alpha=0.15, label="+/- 1 std")
ax.set_xlabel("fold (origin advances by 14 days per fold)")
ax.set_ylabel("MAE (visitors)")
ax.set_title("Figure 4.2: the held-out estimate is stable, not a lucky window")
ax.legend(loc="upper right")
fig.tight_layout()
plt.show()

## RQ2 — Are the intervals honest? (§3.16)

In [ ]:
conf = STAGES["conformal"]["metrics"]
print(f"target coverage {conf['target_coverage']:.0%} -> empirical {conf['empirical_coverage']:.1%} "
      f"| mean width {conf['mean_interval_width']:.1f} visitors")
print("coverage by demand level (the proposal's warning: average calibration can hide busy-day failures):")
pd.Series(conf["coverage_by_demand_level"]).rename("empirical_coverage").to_frame()

In [ ]:
reading(f"""**Reading.** The production intervals are **stratified by predicted demand level** (Mondrian
ACI): one alpha and one residual window per level, so a heavy-tailed bucket calibrates on its own scale
instead of the quiet-day quantile. Overall: **{STAGES["conformal"]["metrics"]["empirical_coverage"]:.1%}**
coverage at a **narrower** mean width than the global quantile they replaced
({STAGES["conformal"]["metrics"]["mean_interval_width"]:.1f} vs
{STAGES["conformal"]["metrics"]["global_quantile_comparison"]["mean_interval_width"]:.1f} visitors). On the
calibration the procedure directly controls - days *predicted* 60+ - coverage holds at
**{STAGES["conformal"]["metrics"]["coverage_by_predicted_bucket"]["60+"]:.1%}** (global quantile:
{STAGES["conformal"]["metrics"]["global_quantile_comparison"]["coverage_by_demand_level"]["60+"]:.1%}), and
the quiet strata sit near target instead of over-covering. Only
{STAGES["conformal"]["metrics"]["global_fallback_share"]:.1%} of store-days fell back to the global interval.

The residual gap on days whose *realised* demand exceeded 60
({STAGES["conformal"]["metrics"]["coverage_by_demand_level"]["60+"]:.1%}) has a different cause: a busy day
the model mis-predicts as quiet is scored with quiet-bucket bounds. That is a **point-forecast limitation on
rare busy days**, not an interval one - so further tail protection must come from better tail prediction, not
from taxing every quiet store-day with wider intervals.""")

In [ ]:
# Chart 4.3 - coverage by realised level: stratified (production) vs global quantile.
gconf = conf["global_quantile_comparison"]
cov_df = pd.DataFrame({
    "stratified": pd.Series(conf["coverage_by_demand_level"]),
    "global quantile": pd.Series(gconf["coverage_by_demand_level"]),
})
x = np.arange(len(cov_df))
width = 0.36
fig, ax = plt.subplots(figsize=(7.5, 3.8))
ax.bar(x - width / 2, cov_df["stratified"], width, label="stratified (production)")
ax.bar(x + width / 2, cov_df["global quantile"], width, label="global quantile")
ax.axhline(conf["target_coverage"], linestyle="--", linewidth=1, label=f"target ({conf['target_coverage']:.0%})")
for xi, v in zip(x - width / 2, cov_df["stratified"]):
    ax.text(xi, v + 0.02, f"{v:.0%}", ha="center", fontsize=8)
for xi, v in zip(x + width / 2, cov_df["global quantile"]):
    ax.text(xi, v + 0.02, f"{v:.0%}", ha="center", fontsize=8)
ax.set_xticks(x, cov_df.index, fontsize=8)
ax.set_ylim(0, 1.08)
ax.set_ylabel("empirical coverage")
ax.set_title("Figure 4.3: stratification restores calibration where it is controlled")
ax.legend(loc="lower left", fontsize=8)
fig.tight_layout()
plt.show()

## RQ3 — What does uncertainty-aware preparation cost and buy? (§3.17–3.20)

In [ ]:
policy_table = joblib.load(ARTIFACTS / "policy_results.joblib")
headline = STAGES["policy"]["metrics"]["headline"]
print(f"uncertainty-aware vs point-forecast: UD {headline['ud_reduction_vs_point']:.1%} lower, "
      f"PS +{headline['ps_increase_vs_point']:,.0f} visitor-equivalents - the trade-off made visible.")
policy_table

In [ ]:
_point_ud = policy_table.loc[policy_table["policy"].eq("point_forecast"), "unmet_ud"].iloc[0]
reading(f"""**Reading.** Preparing exactly to the point forecast is not neutral: it leaves {_point_ud:,.0f}
visitor-equivalents of unmet demand. The uncertainty-aware policy cuts that exposure by
**{STAGES["policy"]["metrics"]["headline"]["ud_reduction_vs_point"]:.1%}** at the price of
+{STAGES["policy"]["metrics"]["headline"]["ps_increase_vs_point"]:,.0f} visitor-equivalents of surplus - the
trade-off made visible. The frozen historical-average sits between, but its inflexibility leaves both
exposures elevated. Which frontier point is right depends on the site-specific cost of waste versus stock-out
- the dashboard's job is to surface this trade-off, never to hide it (S3.20).""")

In [ ]:
# Chart 4.4 - the PS/UD frontier: every policy is a point, no universal winner.
fig, ax = plt.subplots(figsize=(7, 4.5))
ax.scatter(policy_table["ud_per_day"], policy_table["ps_per_day"], s=90)
for _, r in policy_table.iterrows():
    ax.annotate(
        r["policy"],
        (r["ud_per_day"], r["ps_per_day"]),
        textcoords="offset points",
        xytext=(8, 4),
        fontsize=9,
    )
ax.set_xlabel("unmet-demand exposure per store-day (UD, visitor-equivalents)")
ax.set_ylabel("surplus exposure per store-day (PS, visitor-equivalents)")
ax.set_title("Figure 4.4: the preparation trade-off, made visible")
ax.set_xlim(0, policy_table["ud_per_day"].max() * 1.25)
fig.tight_layout()
plt.show()

## Why does the model predict what it predicts? (§3.24)

In [ ]:
top = pd.DataFrame(STAGES["shap"]["metrics"]["top_features"])
print("disclaimer:", STAGES["shap"]["metrics"]["disclaimer"])
print("median reconstruction error:",
      f"{STAGES['shap']['metrics']['median_relative_reconstruction_error']:.1%} (log-linearisation)")
top.rename(columns={"feature_name": "feature", "mean_abs_shap": "mean |SHAP| (visitors)"})

In [ ]:
# Chart 4.5 - what moves the model, in visitor units.
fig, ax = plt.subplots(figsize=(8, 3.5))
top_sorted = top.sort_values("mean_abs_shap")
ax.barh(top_sorted["feature_name"], top_sorted["mean_abs_shap"])
ax.set_xlabel("mean |SHAP| (visitors)")
ax.set_title("Figure 4.5: predictive attribution - not causal effect")
fig.tight_layout()
plt.show()

## Reproducing this report

| Command | Produces |
| --- | --- |
| `make train` | all stages except backtest, checkpointed to `artifacts/runs/latest` |
| `make train-full` | + rolling-origin backtest |
| `make kaggle-push` / `make kaggle-pull` | same pipeline on a free Kaggle kernel, artifacts pulled back |
| `make colab-train` | same pipeline on a Colab VM via the official Colab CLI |

Same seed + data + code ref => identical metrics (verified across local macOS
and Kaggle Linux runs).